<a href="https://colab.research.google.com/github/pratapritesh74-ctrl/Healthcare/blob/main/Healthcare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Healthcare decision intellegence**

In [1]:

# Import libraries
import os
import numpy as np
from pathlib import Path

# Create project directory
PROJECT = Path("/content/Healthcare")

# Create required folders
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / "outputs").mkdir(parents=True, exist_ok=True)

# Move into project directory
os.chdir(PROJECT)

print("Project directory:", os.getcwd())
print("Folders created successfully!")

print("Project folders:")
print(os.listdir(PROJECT))


Project directory: /content/Healthcare
Folders created successfully!
Project folders:
['outputs', 'data']


# Generation of the dataset

In [2]:
%%writefile generate_data.py

"""Synthetic records for the capstone (all values are invented; no real company data).
Run once from the repository root:  python generate_data.py
It writes data/records.csv and prints:  <rows> <columns> <positive_count> <md5 of the file>
Do not edit anything below the line marked NUMERIC CORE: every number in the brief was
measured on this exact output."""
import csv
import hashlib
import math
import os
import random

# ---------------- DOMAIN SETTINGS (names only; they never touch the numbers) ----------------
DOMAIN = "Healthcare"
ID_PREFIX = "H"
COLUMNS = [
    "record_id",
    "months_registered",
    "missed_visit_count_12m",
    "copay_burden_ratio",
    "avg_bill_inr_thousands",
    "adherence_score",
    "chronic_condition_flag",
    "insured_flag",
    "no_show",
    "care_plan_cost_inr_thousands"
]
# ---------------- NUMERIC CORE (identical in every domain) ----------------
N = 800
SEED = 2508
GROUP_P = [0.40, 0.35, 0.25]          # three hidden groups
MU_C1 = [48.0, 30.0, 14.0]            # tenure in months
MU_C2 = [2.0, 6.0, 4.0]               # count of friction events
MU_C3 = [0.22, 0.45, 0.72]            # ratio in [0, 1]
MU_C4 = [math.log(12.0), math.log(26.0), math.log(15.0)]   # log of an amount
MU_C5 = [62.0, 55.0, 40.0]            # score
P_B1 = [0.30, 0.55, 0.45]
P_B2 = [0.60, 0.50, 0.40]


def normal(rng):
    """One standard-normal draw from two uniform draws (Box-Muller)."""
    u1 = 1.0 - rng.random()
    u2 = rng.random()
    return math.sqrt(-2.0 * math.log(u1)) * math.cos(2.0 * math.pi * u2)


def make_rows():
    rng = random.Random(SEED)
    rows = []
    for i in range(N):
        u = rng.random()
        g = 0 if u < GROUP_P[0] else (1 if u < GROUP_P[0] + GROUP_P[1] else 2)
        c1 = round(max(1.0, MU_C1[g] + 9.0 * normal(rng)), 1)
        c2 = int(max(0, round(MU_C2[g] + 1.6 * normal(rng))))
        c3 = round(min(0.99, max(0.01, MU_C3[g] + 0.12 * normal(rng))), 3)
        c4 = round(math.exp(MU_C4[g] + 0.35 * normal(rng)), 1)
        c5 = round(min(100.0, max(0.0, MU_C5[g] + 10.0 * normal(rng))), 1)
        b1 = 1 if rng.random() < P_B1[g] else 0
        b2 = 1 if rng.random() < P_B2[g] else 0
        eta = (-2.05 + 0.9 * (c3 - 0.45) / 0.25 - 0.7 * (c1 - 30.0) / 15.0
               + 0.35 * (c2 - 4.0) / 2.5 + 0.5 * b1 - 0.45 * b2 - 0.25 * (c5 - 55.0) / 12.0)
        p = 1.0 / (1.0 + math.exp(-eta))
        outcome = 1 if rng.random() < p else 0
        mu = 8.0 * math.exp(0.40 * (math.log(c4) - math.log(20.0)) / 0.35 + 0.25 * (c2 - 4.0) / 2.5
                            + 0.15 * b1 - 0.10 * (c1 - 30.0) / 15.0)
        value = round(mu * math.exp(0.45 * normal(rng) - 0.45 * 0.45 / 2.0), 2)
        rows.append([ID_PREFIX + format(i + 1, "04d"), c1, c2, c3, c4, c5, b1, b2, outcome, value])
    return rows


if __name__ == "__main__":
    data = make_rows()
    os.makedirs("data", exist_ok=True)
    with open(os.path.join("data", "records.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(COLUMNS)
        w.writerows(data)
    with open(os.path.join("data", "records.csv"), "rb") as f:
        digest = hashlib.md5(f.read()).hexdigest()
    print(len(data), len(COLUMNS), sum(r[8] for r in data), digest)

Writing generate_data.py


In [5]:
!cd /content/Healthcare && python generate_data.py

800 10 170 e26b46e82f1a1af44dcabc133050802e


In [6]:
import os

print(os.listdir("/content/Healthcare/data"))

['records.csv']


In [7]:
import pandas as pd

df = pd.read_csv("/content/Healthcare/data/records.csv")

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (800, 10)

Column names:
['record_id', 'months_registered', 'missed_visit_count_12m', 'copay_burden_ratio', 'avg_bill_inr_thousands', 'adherence_score', 'chronic_condition_flag', 'insured_flag', 'no_show', 'care_plan_cost_inr_thousands']

First 5 rows:


,record_id,months_registered,missed_visit_count_12m,copay_burden_ratio,avg_bill_inr_thousands,adherence_score,chronic_condition_flag,insured_flag,no_show,care_plan_cost_inr_thousands
0,H0001,62.2,1,0.561,8.1,65.0,0,0,0,1.92
1,H0002,21.9,6,0.425,31.8,75.8,0,0,0,14.29
2,H0003,27.3,7,0.589,32.6,54.0,0,1,1,7.86
3,H0004,32.6,7,0.195,36.9,69.5,0,0,0,33.96
4,H0005,52.3,3,0.345,6.7,54.2,1,1,0,2.00


In [8]:
print("Number of records:", len(df))

print("\nDataset columns:")
print(df.columns.tolist())

print("\nAverage care plan cost:")
print(df["care_plan_cost_inr_thousands"].mean())

Number of records: 800

Dataset columns:
['record_id', 'months_registered', 'missed_visit_count_12m', 'copay_burden_ratio', 'avg_bill_inr_thousands', 'adherence_score', 'chronic_condition_flag', 'insured_flag', 'no_show', 'care_plan_cost_inr_thousands']

Average care plan cost:
8.938175


# **Calculate standard deviation and confidence interval**

In [9]:
import numpy as np
import math

# Total number of records
n = len(df)

# Calculate mean
mean_value = df["care_plan_cost_inr_thousands"].mean()

# Calculate standard deviation
sd_value = np.std(
    df["care_plan_cost_inr_thousands"],
    ddof=1
)

# Calculate standard error
se_value = sd_value / math.sqrt(n)

# Calculate 95% confidence interval
ci95_low = mean_value - (1.96 * se_value)
ci95_high = mean_value + (1.96 * se_value)

# Display results
print("Number of records:", n)
print("Mean:", mean_value)
print("Standard deviation:", sd_value)
print("Standard error:", se_value)
print("95% CI lower limit:", ci95_low)
print("95% CI upper limit:", ci95_high)

Number of records: 800
Mean: 8.938175
Standard deviation: 9.172347061010857
Standard error: 0.3242914403118638
95% CI lower limit: 8.302563776988746
95% CI upper limit: 9.573786223011252


# **Hypothisis testing**

In [11]:

# TASK 1.2: Hypothesis Testing

# Hypothesized average cost (in thousands)
hypothesized_mean = 8.0

# Significance level
alpha = 0.05

# Calculate Z-statistic
z_stat = (mean_value - hypothesized_mean) / se_value

# Critical value for 95% confidence
critical_value = 1.96

# Decision
reject_h0 = "yes" if abs(z_stat) > critical_value else "no"

# Calculate difference from assumed average
cost_gap = mean_value - hypothesized_mean

# Display results

print("Null Hypothesis: Mean = 8.0")
print("Alternative Hypothesis: Mean != 8.0")

print("Significance Level:", alpha)
print("Z-statistic:", z_stat)
print("Critical Value:", critical_value)
print("Reject H0:", reject_h0)
print("Difference from assumed mean:", cost_gap)


Null Hypothesis: Mean = 8.0
Alternative Hypothesis: Mean != 8.0
Significance Level: 0.05
Z-statistic: 2.8929995780887015
Critical Value: 1.96
Reject H0: yes
Difference from assumed mean: 0.9381749999999993


In [13]:

# TASK 1.3: Sample Size Calculation

# Margin of error
E = 0.5

# Confidence critical value
z_critical = 1.96

# Calculate required sample size
n_required = math.ceil((z_critical * sd_value / E) ** 2)

# Display results
print("TASK 1.3: SAMPLE SIZE")
print("Available records:", n)
print("Required sample size:", n_required)
print("Margin of error:", E)


TASK 1.3: SAMPLE SIZE
Available records: 800
Required sample size: 1293
Margin of error: 0.5


In [15]:

# TASK 1.4: Simple Linear Regression (Closed Form)

# Independent variable (X4)
X = df["avg_bill_inr_thousands"].to_numpy()

# Dependent variable (Y)
Y = df["care_plan_cost_inr_thousands"].to_numpy()

# Calculate means
X_mean = np.mean(X)
Y_mean = np.mean(Y)

# Calculate slope using sample covariance / sample variance
reg_slope = np.sum((X - X_mean) * (Y - Y_mean)) / np.sum((X - X_mean) ** 2)

# Calculate intercept
reg_intercept = Y_mean - reg_slope * X_mean

# Calculate predicted values
fitted = reg_slope * X + reg_intercept

# Calculate TSS, ESS and RSS
tss = np.sum((Y - Y_mean) ** 2)
ess = np.sum((fitted - Y_mean) ** 2)
rss = np.sum((Y - fitted) ** 2)

# Check decomposition
tss_equals_ess_plus_rss = (
    "yes" if abs(tss - ess - rss) < 1e-6 * tss else "no"
)

# Calculate R-squared
r2 = 1 - (rss / tss)

# Calculate RMSE and MAE
rmse = np.sqrt(rss / len(Y))
mae = np.mean(np.abs(Y - fitted))

# Display results
print("SIMPLE LINEAR REGRESSION")
print("Regression slope:", reg_slope)
print("Regression intercept:", reg_intercept)
print("TSS:", tss)
print("ESS:", ess)
print("RSS:", rss)
print("TSS = ESS + RSS:", tss_equals_ess_plus_rss)
print("R-squared:", r2)
print("RMSE:", rmse)
print("MAE:", mae)


SIMPLE LINEAR REGRESSION
Regression slope: 0.6701116692566309
Regression intercept: -3.65975685410735
TSS: 67221.42853549999
ESS: 34925.608815222484
RSS: 32295.819720277523
TSS = ESS + RSS: yes
R-squared: 0.5195606457065707
RMSE: 6.35372132300016
MAE: 3.7636036913049806


In [16]:
# TASK 1.5: Gradient Descent

# Standardize X
X_std = (X - np.mean(X)) / np.std(X, ddof=1)

# Initialize parameters
m = 0.0
c = 0.0

# Learning rate and iterations
learning_rate = 0.1
iterations = 500
n = len(Y)

# Store cost values
cost_values = {}

for i in range(1, iterations + 1):

    # Predictions
    Y_pred = m * X_std + c

    # Error
    error = Y - Y_pred

    # Gradients
    dm = (-2/n) * np.sum(X_std * error)
    dc = (-2/n) * np.sum(error)

    # Update parameters
    m = m - learning_rate * dm
    c = c - learning_rate * dc

    # Calculate cost
    J = np.mean((Y - (m * X_std + c))**2)

    if i in [1, 10, 100, 500]:
        cost_values[i] = J

# Convert coefficients back to original X scale
X_sd = np.std(X, ddof=1)
gd_slope = m / X_sd
gd_intercept = c - (m * np.mean(X) / X_sd)

# Compare with closed-form regression
gd_slope_gap = abs(gd_slope - reg_slope)

print("GRADIENT DESCENT RESULTS")
print("J after 1 iteration:", cost_values[1])
print("J after 10 iterations:", cost_values[10])
print("J after 100 iterations:", cost_values[100])
print("J after 500 iterations:", cost_values[500])

print("Gradient Descent Slope:", gd_slope)
print("Gradient Descent Intercept:", gd_intercept)
print("Slope Gap:", gd_slope_gap)

GRADIENT DESCENT RESULTS
J after 1 iteration: 119.45794952709569
J after 10 iterations: 41.79734109475494
J after 100 iterations: 40.369774650346905
J after 500 iterations: 40.369774650346905
Gradient Descent Slope: 0.6701116692566306
Gradient Descent Intercept: -3.6597568541073446
Slope Gap: 3.3306690738754696e-16


In [17]:
# TASK 1.6: Residual Funnel and Log Transformation

# Step 1: Calculate residuals from raw regression
residuals = Y - fitted

# Step 2: Sort by fitted values
sorted_indices = np.argsort(fitted, kind="stable")

# Divide into lower and upper 400 observations
lower_indices = sorted_indices[:400]
upper_indices = sorted_indices[400:]

# Calculate residual standard deviations
resid_sd_lower_raw = np.std(residuals[lower_indices], ddof=1)
resid_sd_upper_raw = np.std(residuals[upper_indices], ddof=1)

resid_sd_ratio_raw = resid_sd_upper_raw / resid_sd_lower_raw

# Step 3: Log-log regression
log_X = np.log(X)
log_Y = np.log(Y)

log_X_mean = np.mean(log_X)
log_Y_mean = np.mean(log_Y)

loglog_slope = np.sum(
    (log_X - log_X_mean) * (log_Y - log_Y_mean)
) / np.sum((log_X - log_X_mean)**2)

loglog_intercept = log_Y_mean - loglog_slope * log_X_mean

log_fitted = loglog_intercept + loglog_slope * log_X

# Step 4: Calculate log-log regression metrics
log_tss = np.sum((log_Y - log_Y_mean)**2)
log_rss = np.sum((log_Y - log_fitted)**2)

loglog_r2 = 1 - log_rss / log_tss
loglog_rmse = np.sqrt(log_rss / len(log_Y))

# Step 5: Calculate log residuals
log_residuals = log_Y - log_fitted

log_sorted_indices = np.argsort(log_fitted, kind="stable")

log_lower_indices = log_sorted_indices[:400]
log_upper_indices = log_sorted_indices[400:]

resid_sd_lower_log = np.std(
    log_residuals[log_lower_indices], ddof=1
)

resid_sd_upper_log = np.std(
    log_residuals[log_upper_indices], ddof=1
)

resid_sd_ratio_log = resid_sd_upper_log / resid_sd_lower_log

# Step 6: Print results
print("RAW REGRESSION RESIDUALS")
print("Lower residual SD:", resid_sd_lower_raw)
print("Upper residual SD:", resid_sd_upper_raw)
print("Raw residual SD ratio:", resid_sd_ratio_raw)

print("\nLOG-LOG REGRESSION")
print("Log-log intercept:", loglog_intercept)
print("Log-log slope:", loglog_slope)
print("Log-log R-squared:", loglog_r2)
print("Log-log RMSE:", loglog_rmse)

print("\nLOG RESIDUALS")
print("Lower residual SD:", resid_sd_lower_log)
print("Upper residual SD:", resid_sd_upper_log)
print("Log residual SD ratio:", resid_sd_ratio_log)

RAW REGRESSION RESIDUALS
Lower residual SD: 2.3155277774036405
Upper residual SD: 8.689129576037987
Raw residual SD ratio: 3.7525481926116

LOG-LOG REGRESSION
Log-log intercept: -2.4370235981975874
Log-log slope: 1.503040417459227
Log-log R-squared: 0.6565229138483721
Log-log RMSE: 0.5267513765274645

LOG RESIDUALS
Lower residual SD: 0.5164599245788176
Upper residual SD: 0.5369198897400375
Log residual SD ratio: 1.0396157846669425


In [18]:
# TASK 1.7: Multiple Linear Regression using Normal Equation

# Dependent variable: log of care plan cost
Y_multi = np.log(df["care_plan_cost_inr_thousands"].to_numpy())

# Independent variables
X4_log = np.log(df["avg_bill_inr_thousands"].to_numpy())

X1 = df["months_registered"].to_numpy()
X2 = df["missed_visit_count_12m"].to_numpy()
X3 = df["copay_burden_ratio"].to_numpy()
X5 = df["adherence_score"].to_numpy()
X6 = df["chronic_condition_flag"].to_numpy()
X7 = df["insured_flag"].to_numpy()

# Create design matrix in the required order
A = np.column_stack((
    np.ones(len(df)),
    X4_log,
    X1,
    X2,
    X3,
    X5,
    X6,
    X7
))

# Normal Equation
beta = np.linalg.solve(A.T @ A, A.T @ Y_multi)

# Predictions
Y_multi_pred = A @ beta

# Calculate R-squared
mr_tss = np.sum((Y_multi - np.mean(Y_multi))**2)
mr_rss = np.sum((Y_multi - Y_multi_pred)**2)

mr_r2 = 1 - (mr_rss / mr_tss)

# Adjusted R-squared
n = len(Y_multi)
p = 7

mr_adj_r2 = 1 - (1 - mr_r2) * (n - 1) / (n - p - 1)

# Adjusted R-squared for Task 1.6 log-log model
loglog_adj_r2 = 1 - (1 - loglog_r2) * (n - 1) / (n - 1 - 1)

# Print results
print("MULTIPLE LINEAR REGRESSION")

for i, value in enumerate(beta):
    print(f"mr_beta_{i}:", value)

print("mr_r2:", mr_r2)
print("mr_adj_r2:", mr_adj_r2)
print("loglog_adj_r2:", loglog_adj_r2)

MULTIPLE LINEAR REGRESSION
mr_beta_0: -1.8612944243241323
mr_beta_1: 1.157556334061802
mr_beta_2: -0.0074014553417748215
mr_beta_3: 0.09402389612412261
mr_beta_4: 0.10671478964597791
mr_beta_5: 0.0027989136208992544
mr_beta_6: 0.15178892336736097
mr_beta_7: 0.017199096070880403
mr_r2: 0.7425299005162278
mr_adj_r2: 0.7402542809500834
loglog_adj_r2: 0.6560924914346482


# **Part1.py**

In [19]:
%%writefile /content/Healthcare/part1.py

# PharmEasy Capstone - Part 1
# Estimation, Inference and Regression

import numpy as np
import pandas as pd
import math

# Load dataset
df = pd.read_csv("/content/Healthcare/data/records.csv")

print("Dataset loaded successfully")
print("Number of records:", len(df))

Writing /content/Healthcare/part1.py


In [20]:
import os

print(os.listdir("/content/Healthcare"))

['outputs', 'data', 'part1.py', 'generate_data.py']


In [21]:
from google.colab import files

files.download('/content/Healthcare/generate_data.py')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
from google.colab import files

files.download('/content/Healthcare/part1.py')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
from google.colab import files

files.download('/content/Healthcare/data/records.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:

%%writefile /content/Healthcare/part1.py

# PharmEasy Healthcare Decision Intelligence
# Part 1: Estimation, Inference and Regression

import numpy as np
import pandas as pd
import math
from pathlib import Path

# --------------------------------------------------
# LOAD DATASET
# --------------------------------------------------

BASE_DIR = Path(__file__).resolve().parent
df = pd.read_csv(BASE_DIR / "data" / "records.csv")

n = len(df)
results = {}

# --------------------------------------------------
# TASK 1.1: ESTIMATION
# --------------------------------------------------

Y = df["care_plan_cost_inr_thousands"].to_numpy()

mean_value = np.mean(Y)
sd_value = np.std(Y, ddof=1)
se_value = sd_value / np.sqrt(n)

ci95_low = mean_value - 1.96 * se_value
ci95_high = mean_value + 1.96 * se_value

results["n"] = n
results["mean_value"] = mean_value
results["sd_value"] = sd_value
results["se_value"] = se_value
results["ci95_low"] = ci95_low
results["ci95_high"] = ci95_high

# --------------------------------------------------
# TASK 1.2: Z-TEST
# --------------------------------------------------

hypothesized_mean = 8.0
z_stat = (mean_value - hypothesized_mean) / se_value
reject_h0 = "yes" if abs(z_stat) > 1.96 else "no"

results["z_stat"] = z_stat
results["reject_h0"] = reject_h0

# --------------------------------------------------
# TASK 1.3: SAMPLE SIZE
# --------------------------------------------------

E = 0.5
n_required = math.ceil((1.96 * sd_value / E) ** 2)

results["n_required"] = n_required

# --------------------------------------------------
# TASK 1.4: SIMPLE LINEAR REGRESSION
# --------------------------------------------------

X = df["avg_bill_inr_thousands"].to_numpy()

X_mean = np.mean(X)
Y_mean = np.mean(Y)

reg_slope = np.sum((X - X_mean) * (Y - Y_mean)) / np.sum((X - X_mean) ** 2)
reg_intercept = Y_mean - reg_slope * X_mean

fitted = reg_slope * X + reg_intercept

tss = np.sum((Y - Y_mean) ** 2)
ess = np.sum((fitted - Y_mean) ** 2)
rss = np.sum((Y - fitted) ** 2)

tss_equals_ess_plus_rss = (
    "yes" if abs(tss - ess - rss) < 1e-6 * tss else "no"
)

r2 = 1 - rss / tss
rmse = np.sqrt(rss / n)
mae = np.mean(np.abs(Y - fitted))

results["reg_slope"] = reg_slope
results["reg_intercept"] = reg_intercept
results["tss"] = tss
results["ess"] = ess
results["rss"] = rss
results["tss_equals_ess_plus_rss"] = tss_equals_ess_plus_rss
results["r2"] = r2
results["rmse"] = rmse
results["mae"] = mae

# --------------------------------------------------
# TASK 1.5: GRADIENT DESCENT
# --------------------------------------------------

X_sd = np.std(X, ddof=1)
X_std = (X - X_mean) / X_sd

m = 0.0
c = 0.0
learning_rate = 0.1

# Initial cost before updates
gd_J0 = np.mean((Y - (m * X_std + c)) ** 2)

cost_values = {}

for i in range(1, 501):
    Y_pred = m * X_std + c
    error = Y - Y_pred

    dm = (-2 / n) * np.sum(X_std * error)
    dc = (-2 / n) * np.sum(error)

    m = m - learning_rate * dm
    c = c - learning_rate * dc

    J = np.mean((Y - (m * X_std + c)) ** 2)

    if i in [1, 10, 100, 500]:
        cost_values[i] = J

gd_slope = m / X_sd
gd_intercept = c - (m * X_mean / X_sd)

gd_slope_gap = abs(gd_slope - reg_slope)

results["gd_J0"] = gd_J0
results["gd_J1"] = cost_values[1]
results["gd_J10"] = cost_values[10]
results["gd_J100"] = cost_values[100]
results["gd_J500"] = cost_values[500]
results["gd_slope"] = gd_slope
results["gd_intercept"] = gd_intercept
results["gd_slope_gap"] = gd_slope_gap

# --------------------------------------------------
# TASK 1.6: RESIDUAL FUNNEL AND LOG TRANSFORMATION
# --------------------------------------------------

residuals = Y - fitted

sorted_indices = np.argsort(fitted, kind="stable")
lower_indices = sorted_indices[:400]
upper_indices = sorted_indices[400:]

resid_sd_lower_raw = np.std(residuals[lower_indices], ddof=1)
resid_sd_upper_raw = np.std(residuals[upper_indices], ddof=1)
resid_sd_ratio_raw = resid_sd_upper_raw / resid_sd_lower_raw

# Log-log regression
log_X = np.log(X)
log_Y = np.log(Y)

log_X_mean = np.mean(log_X)
log_Y_mean = np.mean(log_Y)

loglog_slope = (
    np.sum((log_X - log_X_mean) * (log_Y - log_Y_mean))
    / np.sum((log_X - log_X_mean) ** 2)
)

loglog_intercept = log_Y_mean - loglog_slope * log_X_mean
log_fitted = loglog_intercept + loglog_slope * log_X

log_tss = np.sum((log_Y - log_Y_mean) ** 2)
log_rss = np.sum((log_Y - log_fitted) ** 2)

loglog_r2 = 1 - log_rss / log_tss
loglog_rmse = np.sqrt(log_rss / n)

log_residuals = log_Y - log_fitted

log_sorted_indices = np.argsort(log_fitted, kind="stable")
log_lower_indices = log_sorted_indices[:400]
log_upper_indices = log_sorted_indices[400:]

resid_sd_lower_log = np.std(log_residuals[log_lower_indices], ddof=1)
resid_sd_upper_log = np.std(log_residuals[log_upper_indices], ddof=1)
resid_sd_ratio_log = resid_sd_upper_log / resid_sd_lower_log

results["resid_sd_lower_raw"] = resid_sd_lower_raw
results["resid_sd_upper_raw"] = resid_sd_upper_raw
results["resid_sd_ratio_raw"] = resid_sd_ratio_raw
results["loglog_intercept"] = loglog_intercept
results["loglog_slope"] = loglog_slope
results["loglog_r2"] = loglog_r2
results["loglog_rmse"] = loglog_rmse
results["resid_sd_lower_log"] = resid_sd_lower_log
results["resid_sd_upper_log"] = resid_sd_upper_log
results["resid_sd_ratio_log"] = resid_sd_ratio_log

# --------------------------------------------------
# TASK 1.7: MULTIPLE LINEAR REGRESSION
# --------------------------------------------------

X1 = df["months_registered"].to_numpy()
X2 = df["missed_visit_count_12m"].to_numpy()
X3 = df["copay_burden_ratio"].to_numpy()
X4_log = np.log(df["avg_bill_inr_thousands"].to_numpy())
X5 = df["adherence_score"].to_numpy()
X6 = df["chronic_condition_flag"].to_numpy()
X7 = df["insured_flag"].to_numpy()

A = np.column_stack((
    np.ones(n),
    X4_log,
    X1,
    X2,
    X3,
    X5,
    X6,
    X7
))

beta = np.linalg.solve(A.T @ A, A.T @ log_Y)

Y_multi_pred = A @ beta

mr_tss = np.sum((log_Y - np.mean(log_Y)) ** 2)
mr_rss = np.sum((log_Y - Y_multi_pred) ** 2)

mr_r2 = 1 - mr_rss / mr_tss

p = 7
mr_adj_r2 = 1 - (1 - mr_r2) * (n - 1) / (n - p - 1)

loglog_adj_r2 = 1 - (1 - loglog_r2) * (n - 1) / (n - 2)

for i in range(8):
    results[f"mr_beta_{i}"] = beta[i]

results["mr_r2"] = mr_r2
results["mr_adj_r2"] = mr_adj_r2
results["loglog_adj_r2"] = loglog_adj_r2

# --------------------------------------------------
# SAVE RESULTS
# --------------------------------------------------

output_dir = BASE_DIR / "outputs"
output_dir.mkdir(exist_ok=True)

output_file = output_dir / "part1_results.txt"

with open(output_file, "w") as f:
    for key, value in results.items():
        if isinstance(value, (int, np.integer)):
            f.write(f"{key} = {int(value)}\n")
        elif isinstance(value, str):
            f.write(f"{key} = {value}\n")
        else:
            f.write(f"{key} = {float(value):.4f}\n")

print("Part 1 completed successfully!")
print("Total records:", n)
print("Total result keys:", len(results))
print("Results saved to:", output_file)
print("\nMULTIPLE REGRESSION SUMMARY")
print("mr_r2:", round(mr_r2, 4))
print("mr_adj_r2:", round(mr_adj_r2, 4))


Overwriting /content/Healthcare/part1.py


In [25]:
!cd /content/Healthcare && python part1.py

Part 1 completed successfully!
Total records: 800
Total result keys: 47
Results saved to: /content/Healthcare/outputs/part1_results.txt

MULTIPLE REGRESSION SUMMARY
mr_r2: 0.7425
mr_adj_r2: 0.7403


In [26]:
from google.colab import files

files.download('/content/Healthcare/part1.py')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
files.download('/content/Healthcare/outputs/part1_results.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [28]:
import numpy
import pandas

print("numpy==" + numpy.__version__)
print("pandas==" + pandas.__version__)

numpy==2.4.6
pandas==2.2.3


In [29]:
import numpy
import pandas

print("numpy==" + numpy.__version__)
print("pandas==" + pandas.__version__)

numpy==2.4.6
pandas==2.2.3
